Assignment 3

Conor Murray

In [14]:
import numpy as np
import pandas as pd

In [15]:
df = pd.read_csv("k-means_question_sample_data.csv", header=None)
data = df.values
print(data.shape)
df.head()

(100, 5)


,0,1,2,3,4
0,-6.39561,9.74967,-6.33799,5.09486,-3.95324
1,-6.50576,7.25495,-8.91225,-8.96219,-4.02736
2,-6.47878,6.88525,-9.45403,-8.96097,-4.31427
3,-2.54190,-1.20981,8.29463,-4.89130,1.20691
4,-6.12204,9.59694,-6.33487,5.08625,-3.92635


In [16]:
def standardizer(arr):
    n = arr.shape[0]
    m = arr.shape[1]
    result = np.zeros((n, m))
    for j in range(m):
        col = arr[:, j]
        colMean = np.mean(col)
        colStd = np.std(col)
        for i in range(n):
            if colStd == 0:
                result[i, j] = 0
            else:
                result[i, j] = (arr[i, j] - colMean) / colStd
    return result
stdData = standardizer(data)
print(stdData[0:5])
print(np.mean(stdData, axis=0))
print(np.std(stdData, axis=0))

[[-0.61458461  0.98095786 -0.5242401   1.37080724 -0.6369283 ]
 [-0.67281668  0.43500148 -0.86578446 -1.02516486 -0.66805991]
 [-0.65855339  0.35409457 -0.93766605 -1.02495691 -0.7885668 ]
 [ 1.42272391 -1.41746683  1.41716784 -0.33129674  1.53041906]
 [-0.46995865  0.9475337  -0.52382615  1.3693397  -0.62563406]]
[-3.13082893e-16 -4.44089210e-17 -5.77315973e-17  7.88258347e-17
 -9.10382880e-17]
[1. 1. 1. 1. 1.]


In [17]:
def calculateDistance(obj1, obj2):
    total = 0
    for j in range(len(obj1)):
        diff = obj1[j] - obj2[j]
        total = total + diff * diff
    distance = np.sqrt(total)
    return distance
print(calculateDistance(stdData[0], stdData[1]))
print(calculateDistance(stdData[0], stdData[4]))

2.481887059216075
0.1488748678601796


In [18]:
def findClosestCentroid(distArray):
    closestIndex = 0
    smallestDist = distArray[0]
    for i in range(1, len(distArray)):
        if distArray[i] < smallestDist:
            smallestDist = distArray[i]
            closestIndex = i
    return closestIndex
print(findClosestCentroid(np.array([4.2, 1.7, 3.9])))

1


In [19]:
def calculateCentroids(assignmentArr):
    n = assignmentArr.shape[0]
    m = assignmentArr.shape[1]
    centroid = np.zeros(m)
    for j in range(m):
        total = 0
        for i in range(n):
            total = total + assignmentArr[i, j]
        centroid[j] = total / n
    return centroid
print(calculateCentroids(stdData[0:3]))

[-0.64865156  0.59001797 -0.77589687 -0.22643818 -0.69785167]


In [20]:
def main(arr, k, maxIter=100, seed=42):
    stdArr = standardizer(arr)
    n = stdArr.shape[0]
    m = stdArr.shape[1]
    np.random.seed(seed)
    startIndexes = np.random.choice(n, k, replace=False)
    centroids = np.zeros((k, m))
    for c in range(k):
        centroids[c] = stdArr[startIndexes[c]]
    assignments = np.zeros(n, dtype=int)
    iteration = 0
    while iteration < maxIter:
        iteration = iteration + 1
        for i in range(n):
            distArray = np.zeros(k)
            for c in range(k):
                distArray[c] = calculateDistance(stdArr[i], centroids[c])
            assignments[i] = findClosestCentroid(distArray)
        newCentroids = np.zeros((k, m))
        for c in range(k):
            mask = assignments == c
            assignmentArr = stdArr[mask]
            if assignmentArr.shape[0] == 0:
                newCentroids[c] = centroids[c]
            else:
                newCentroids[c] = calculateCentroids(assignmentArr)
        changed = False
        for c in range(k):
            if calculateDistance(centroids[c], newCentroids[c]) > 0:
                changed = True
        centroids = newCentroids
        if changed == False:
            break
    sse = 0
    for i in range(n):
        d = calculateDistance(stdArr[i], centroids[assignments[i]])
        sse = sse + d * d
    return assignments, centroids, iteration, sse

In [21]:
k = 3
assignments, centroids, iterations, sse = main(data, k)
print("Iterations until convergence:", iterations)
print("SSE:", sse)
print("Final centroids (standardized):")
print(centroids)
for c in range(k):
    count = 0
    for i in range(len(assignments)):
        if assignments[i] == c:
            count = count + 1
    print("Cluster", c, "size:", count)

Iterations until convergence: 3
SSE: 1.9638596911035333
Final centroids (standardized):
[[ 1.41781649 -1.37784757  1.40776415 -0.32365599  1.41900187]
 [-0.68793252  0.37542498 -0.87626756 -1.01206075 -0.74949762]
 [-0.70903753  0.99104607 -0.50494304  1.36638525 -0.64679219]]
Cluster 0 size: 33
Cluster 1 size: 34
Cluster 2 size: 33
